# BitterTruth-AI -- ARC-AGI-3 Competition Submission

**Architecture**: Solver-free cognitive pipeline -- OBSERVE > CLASSIFY > EXTRACT_GOAL > MAP_EFFECTS > PLAN > EXECUTE > VERIFY

**Key innovations**:
- H53: Win-state goal templates (learn goals from prior sessions)
- H55: Self-toggle rule extrapolation (plan without visiting every cell)
- H56: Rule-based strategy unlock (escape explore-forever deadlock)
- H51: Autonomous spatial navigation + dead reckoning for movement games
- H47: Score-correlated goal discovery

**No solver data required** -- the system discovers mechanics autonomously.

In [ ]:
# -- Install ARC-AGI SDK from competition-provided wheels ----------
import subprocess, sys, os, glob

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    # Discover what's mounted (Kaggle may nest under competitions/ and datasets/)
    input_dir = '/kaggle/input'
    print('Mounted inputs:')
    try:
        for root, dirs, files in os.walk(input_dir):
            depth = root.replace(input_dir, '').count(os.sep)
            if depth <= 2:
                indent = '  ' * (depth + 1)
                print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    except Exception as e:
        print(f'  ERROR walking {input_dir}: {e}')

    # Search for wheels directory anywhere under /kaggle/input
    wheels_dir = None
    for pattern in [
        '/kaggle/input/**/arc_agi_3_wheels',
        '/kaggle/input/arc_agi_3_wheels',
    ]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            wheels_dir = matches[0]
            break

    if wheels_dir:
        print(f'Installing SDK wheels from {wheels_dir}')
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-index',
             '--find-links', wheels_dir, 'arc-agi', 'arcengine'],
            capture_output=True, text=True
        )
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.returncode != 0:
            print('STDERR:', result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print('WARNING: arc_agi_3_wheels directory not found anywhere under /kaggle/input')
        print('Attempting pip install with internet (may fail if offline)...')
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', 'arc-agi', 'arcengine'],
            capture_output=True, text=True
        )
        print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
        if result.returncode != 0:
            print('STDERR:', result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
else:
    print('Local dev -- skipping wheel install (assumes arc-agi is already installed)')

In [ ]:
# -- Environment setup ---------------------------------------------
import os, sys, time, json, logging, glob

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

# Suppress verbose logging -- keep output clean
logging.basicConfig(level=logging.WARNING)
for noisy in ['arc_agi', 'arcengine', 'engines', 'rungs', 'cognitive_loop']:
    logging.getLogger(noisy).setLevel(logging.ERROR)

def _find_dir(name, fallback=None):
    """Search /kaggle/input recursively for a directory by name."""
    matches = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if matches:
        return matches[0]
    # Also try direct path
    direct = f'/kaggle/input/{name}'
    if os.path.isdir(direct):
        return direct
    return fallback

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    # Find code dataset
    CODE_DIR = _find_dir('bittertruth-ai', '/kaggle/input/bittertruth-ai')
    # Find knowledge dataset
    KNOWLEDGE_DIR = _find_dir('bittertruth-knowledge', '/kaggle/input/bittertruth-knowledge')
    # Add code dir to path
    sys.path.insert(0, CODE_DIR)
    # Find environment_files for competition games
    ENVS_DIR = _find_dir('environment_files',
                         '/kaggle/input/arc-prize-2026-arc-agi-3/environment_files')
else:
    CODE_DIR = os.path.dirname(os.path.abspath('.'))
    ENVS_DIR = 'environment_files'
    KNOWLEDGE_DIR = 'competition_knowledge'

print(f'Running on: {"Kaggle" if KAGGLE else "Local"}')
print(f'Code:  {CODE_DIR} (exists={os.path.isdir(CODE_DIR)})')
print(f'Games: {ENVS_DIR} (exists={os.path.isdir(ENVS_DIR)})')
print(f'Knowledge: {KNOWLEDGE_DIR} (exists={os.path.isdir(KNOWLEDGE_DIR)})')

In [ ]:
# -- SDK + cognitive system imports --------------------------------
# Strategy: bypass ALL __init__.py files (they have heavy cascading imports)
# and load ONLY the specific .py files we need via importlib.
import types, importlib, importlib.util

# Verify CODE_DIR contents
if KAGGLE:
    print(f'CODE_DIR contents: {sorted(os.listdir(CODE_DIR))[:20]}')
    engines_path = os.path.join(CODE_DIR, 'engines')
    if os.path.isdir(engines_path):
        print(f'engines/ contents: {sorted(os.listdir(engines_path))[:15]}')

# Import SDK (these are properly installed packages)
from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState
print('SDK imports OK')

# -- Module shimming: register stub packages in sys.modules so that
#    "from engines.cognition.X import Y" resolves to our hand-loaded
#    modules WITHOUT triggering engines/__init__.py cascading imports.

def _make_stub_package(dotted_name):
    """Create an empty package module in sys.modules."""
    if dotted_name in sys.modules:
        return
    mod = types.ModuleType(dotted_name)
    mod.__path__ = [os.path.join(CODE_DIR, *dotted_name.split('.'))]
    mod.__package__ = dotted_name
    mod.__spec__ = None
    sys.modules[dotted_name] = mod

def _load_module(dotted_name, rel_path):
    """Load a .py file via importlib and register it in sys.modules."""
    full_path = os.path.join(CODE_DIR, *rel_path.split('/'))
    if not os.path.isfile(full_path):
        raise FileNotFoundError(f'{full_path} not found')
    spec = importlib.util.spec_from_file_location(dotted_name, full_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[dotted_name] = mod  # Register BEFORE exec (handles circular refs)
    spec.loader.exec_module(mod)
    # Also attach to parent package for "from X.Y import Z" resolution
    parts = dotted_name.rsplit('.', 1)
    if len(parts) == 2 and parts[0] in sys.modules:
        setattr(sys.modules[parts[0]], parts[1], mod)
    return mod

# Step 1: Create stub packages (no code executed, just empty shells)
for pkg in ['config', 'engines', 'engines.cognition', 'engines.perception',
            'engines.self_model', 'engines.social', 'engines.memory',
            'engines.consciousness', 'engines.regulation', 'engines.planning']:
    _make_stub_package(pkg)
print('Stub packages registered')

# Step 2: Load modules in dependency order
# config.cognitive_parameters (needed by phenomenology_layer)
_load_module('config.cognitive_parameters', 'config/cognitive_parameters.py')
print('  config.cognitive_parameters loaded')

# perception (needed by cognitive_loop)
_load_module('engines.perception.perceptual_field', 'engines/perception/perceptual_field.py')
_load_module('engines.perception.perceiver', 'engines/perception/perceiver.py')
print('  perception modules loaded')

# cognition (needed by cognitive_loop)
_load_module('engines.cognition.causal_map', 'engines/cognition/causal_map.py')
_load_module('engines.cognition.cognitive_frame', 'engines/cognition/cognitive_frame.py')
_load_module('engines.cognition.phenomenology_layer', 'engines/cognition/phenomenology_layer.py')
print('  cognition modules loaded')

# cognitive_loop (top-level, uses all above)
_load_module('cognitive_loop', 'cognitive_loop.py')
print('  cognitive_loop loaded')

# Get the classes we need
CausalMap = sys.modules['engines.cognition.causal_map'].CausalMap
CognitiveLoop = sys.modules['cognitive_loop'].CognitiveLoop

print(f'All imports OK  (CausalMap={CausalMap.__name__}, CognitiveLoop={CognitiveLoop.__name__})')


In [ ]:
# -- Initialize arcade in OFFLINE mode ----------------------------
# OFFLINE = reads from local environment_files/, no API calls needed.
# This satisfies the Kaggle "no internet" constraint.
arcade = Arcade(
    operation_mode=OperationMode.OFFLINE,
    arc_api_key='',          # Not needed offline
    environments_dir=ENVS_DIR,
)

games = arcade.get_environments()
print(f'Available games: {len(games)}')
for g in games:
    print(f'  {g.game_id}  tags={g.tags}')

In [ ]:
# -- Load pre-built knowledge --------------------------------------
# During training (evolution runs), the cognitive agent accumulated
# effects, rules, color cycles, and win-state templates.
# These are exported to JSON and bundled as a Kaggle Dataset.
# On competition day, we load them to warm-start each game.

BUNDLED_KNOWLEDGE = {}  # game_id -> knowledge dict

if os.path.exists(KNOWLEDGE_DIR):
    for fname in os.listdir(KNOWLEDGE_DIR):
        if fname.endswith('.json'):
            game_id = fname.replace('.json', '')
            try:
                with open(os.path.join(KNOWLEDGE_DIR, fname)) as f:
                    BUNDLED_KNOWLEDGE[game_id] = json.load(f)
                print(f'  Loaded knowledge: {game_id}')
            except Exception as e:
                print(f'  Warning: could not load {fname}: {e}')
else:
    print('No bundled knowledge directory found -- starting cold')

print(f'Knowledge loaded for {len(BUNDLED_KNOWLEDGE)} games')

In [ ]:
# -- Core agent: play one game session ----------------------------

def play_game(game_id: str, scorecard_id: str, knowledge: dict = None,
              verbose: bool = True, knowledge_export_dir: str = None,
              t_budget: float = 300.0, **_kwargs) -> dict:
    """
    AGI learning loop: keep playing the same game until t_budget is exhausted.
    Knowledge accumulates across every attempt -- causal map, goal cells,
    winning sequences -- so the agent gets smarter with each replay.
    """
    game_start = time.time()
    best_score = 0.0
    best_levels = 0
    best_actions = 0
    win_levels = 1  # will be discovered on first attempt

    # Accumulated knowledge grows with every scored attempt
    accumulated = dict(knowledge) if knowledge else {}
    loop = None        # same loop object reused; causal map persists across attempts
    attempt = 0
    last_print = game_start

    while True:
        elapsed = time.time() - game_start
        remaining = t_budget - elapsed
        if remaining < 1.0:
            break

        attempt += 1
        try:
            env = arcade.make(game_id, scorecard_id=scorecard_id, seed=(attempt - 1) % 16)
            if env is None:
                break

            obs = env.reset()
            available_actions = obs.available_actions
            win_levels = obs.win_levels
            game_type = game_id[:4]

            # First attempt: try replay from accumulated winning sequences
            if attempt == 1 and accumulated.get("winning_sequences"):
                score, levels, acts = _replay_sequences(
                    env, accumulated["winning_sequences"], win_levels, False)
                if score > best_score:
                    best_score, best_levels, best_actions = score, levels, acts
                if verbose:
                    print(f"  [{game_id}] A1 Replay: {levels}/{win_levels} score={score:.3f}")
                if score >= 1.0:
                    break
                continue  # proceed to cognitive on next attempt

            # Cognitive play -- uses & grows accumulated knowledge
            show = verbose and (attempt <= 2 or time.time() - last_print >= 30.0)
            if show:
                last_print = time.time()

            score, levels, acts, loop = _cognitive_play(
                env, game_id, game_type, available_actions, win_levels,
                knowledge=accumulated,
                verbose=show,
                t_budget=remaining - 0.5,   # let game use all remaining time
                prior_loop=loop,
            )

            if score > best_score:
                best_score, best_levels, best_actions = score, levels, acts

            if show:
                print(f"  [{game_id}] A{attempt}: {levels}/{win_levels} levels "                      f"score={score:.3f} best={best_score:.3f} "                      f"elapsed={elapsed:.0f}s")

            # After a scoring attempt: export + merge winning sequences
            if score > 0 and knowledge_export_dir and loop and loop._causal_map:
                try:
                    _export_knowledge(loop._causal_map, game_id, knowledge_export_dir)
                    path = os.path.join(knowledge_export_dir, f"{game_id}.json")
                    if os.path.exists(path):
                        with open(path) as _f:
                            learned = json.load(_f)
                        for lvl, seq in learned.get("winning_sequences", {}).items():
                            accumulated.setdefault("winning_sequences", {})[lvl] = seq
                        accumulated["game_id"] = game_id
                except Exception:
                    pass

            if best_score >= 1.0:
                break   # perfect score -- done

            # Early-exit: stop wasting time on games that never score.
            # 10s/attempt × 12 attempts = 120s. If still zero, move on.
            # Games that score even once continue for the full budget.
            if best_score == 0.0 and attempt >= 12 and (time.time() - game_start) > 180:
                if verbose:
                    print(f"  [{game_id}] giving up after {attempt} zero-score attempts ({int(time.time()-game_start)}s)")
                break

        except Exception as e:
            if verbose:
                print(f"  [{game_id}] A{attempt} error: {e}")

    elapsed_total = time.time() - game_start
    if verbose:
        is_move = any(a in (getattr(env, "_available", []) or []) for a in (1,2,3,4))
        print(f"  [{game_id}] Done: {attempt} attempts, "              f"best={best_score:.3f} ({best_levels}/{win_levels}L), {elapsed_total:.0f}s")

    # Final knowledge export
    if knowledge_export_dir and loop and loop._causal_map:
        try:
            _export_knowledge(loop._causal_map, game_id, knowledge_export_dir)
        except Exception:
            pass

    return {"game_id": game_id, "score": best_score, "levels": best_levels,
            "actions": best_actions, "elapsed": elapsed_total, "attempts": attempt}


print("play_game() defined")


In [ ]:
# -- Replay helper -------------------------------------------------

def _replay_sequences(env, sequences_by_level: dict, win_levels: int,
                      verbose: bool) -> tuple:
    """
    Replay pre-recorded winning sequences level by level.
    sequences_by_level: {level_str: [action_entry, ...]}
      where action_entry is int, str "ACTION6", or dict {'action': int, 'data': {...}}
    Returns (score, levels_completed, actions_taken).
    """
    actions_taken = 0
    obs = getattr(env, 'observation_space', None) or env.reset()

    for level_num in range(1, win_levels + 1):
        seq = sequences_by_level.get(str(level_num), [])
        if not seq:
            break

        for seq_entry in seq:
            # Parse entry -- support multiple formats
            if isinstance(seq_entry, dict):
                action_num = seq_entry.get('action', seq_entry.get('action_num', 6))
                action_data = seq_entry.get('data', seq_entry.get('action_data', None))
            elif isinstance(seq_entry, int):
                action_num = seq_entry
                action_data = None
            elif isinstance(seq_entry, str) and seq_entry.startswith('ACTION'):
                action_num = int(seq_entry.replace('ACTION', ''))
                action_data = None
            else:
                action_num = int(seq_entry) if seq_entry else 6
                action_data = None

            action = getattr(GameAction, f'ACTION{action_num}', GameAction.ACTION6)
            try:
                obs = env.step(action, data=action_data)
                actions_taken += 1
                if obs.state == GameState.WIN:
                    levels_completed = getattr(obs, 'levels_completed', win_levels)
                    score = levels_completed / win_levels if win_levels > 0 else 1.0
                    return score, levels_completed, actions_taken
                if obs.state == GameState.GAME_OVER:
                    levels_completed = getattr(obs, 'levels_completed', 0) or 0
                    score = levels_completed / win_levels if win_levels > 0 else 0.0
                    return score, levels_completed, actions_taken
            except Exception:
                break

    levels_completed = getattr(obs, 'levels_completed', 0) or 0
    score = levels_completed / win_levels if win_levels > 0 else 0.0
    return score, levels_completed, actions_taken


print('_replay_sequences() defined')

In [ ]:
# -- Cognitive play helper -----------------------------------------

def _cognitive_play(env, game_id: str, game_type: str, available_actions: list,
                    win_levels: int, knowledge: dict = None,
                    verbose: bool = True, t_budget: float = 300.0,
                    prior_loop=None):
    """
    Run the cognitive pipeline for one game session.
    Critical: snapshot the causal map BEFORE start_game() resets it,
    then restore it after -- so learned effects survive across attempts.
    """
    import numpy as np

    obs = getattr(env, "observation_space", None) or env.reset()
    max_actions = getattr(obs, "max_actions", 500) or 500

    # --- Snapshot learned state before start_game() wipes the causal map ---
    _saved = {}
    if prior_loop is not None and getattr(prior_loop, "_causal_map", None):
        cm = prior_loop._causal_map
        _saved = {
            "effects":      dict(getattr(cm, "_effects", {})),
            "rules":        list(getattr(cm, "_rules", [])),
            "goal_cells":   dict(getattr(cm, "_goal_cells", {})),
            "color_cycles": dict(getattr(cm, "_color_cycles", {})),
            "win_templates":dict(getattr(cm, "_win_templates", {})),
            "explored":     set(getattr(cm, "_explored", set())),
            "all_positions":set(getattr(cm, "_all_positions", set())),
        }

    if prior_loop is not None:
        loop = prior_loop
        loop.start_game(game_id, available_actions, max_actions)
    else:
        loop = CognitiveLoop(
            decision_system=None, context_builder=None, db=None, verbose=False)
        loop.start_game(game_id, available_actions, max_actions)

    # --- Restore accumulated learned state into fresh causal map ---
    if _saved and getattr(loop, "_causal_map", None):
        cm = loop._causal_map
        cm._effects.update(_saved["effects"])
        for r in _saved["rules"]:
            if r not in cm._rules:
                cm._rules.append(r)
        cm._goal_cells.update(_saved["goal_cells"])
        if hasattr(cm, "_color_cycles"):
            cm._color_cycles.update(_saved["color_cycles"])
        if hasattr(cm, "_win_templates"):
            cm._win_templates.update(_saved["win_templates"])
        cm._explored.update(_saved["explored"])
        cm._all_positions.update(_saved["all_positions"])

    # Warm-start from bundled/accumulated knowledge
    if knowledge and loop._causal_map:
        _inject_knowledge(loop._causal_map, knowledge,
                          game_id=game_id, game_type=game_type)

    actions_taken = 0
    prev_levels = 0
    prev_score = 0.0

    t_game_start = time.time()
    while actions_taken < max_actions:
        if time.time() - t_game_start > t_budget:
            break
        frame = getattr(obs, "frame", None)
        current_available = getattr(obs, "available_actions", None) or available_actions

        try:
            action_num, action_data, cf = loop.cycle(
                frame=frame, obs=obs, available_actions=current_available)
        except Exception as e:
            if verbose:
                print(f"    cycle() error: {e}")
            import random
            action_num = random.choice(current_available)
            action_data = None

        if action_num not in current_available:
            import random
            action_num = random.choice(current_available)
            action_data = None

        action = getattr(GameAction, f"ACTION{action_num}", GameAction.ACTION1)

        try:
            new_obs = env.step(action, data=action_data)
        except Exception as e:
            if verbose:
                print(f"    step() error: {e}")
            break

        if new_obs is None:
            break

        actions_taken += 1
        new_frame = getattr(new_obs, "frame", None)

        frame_changed = False
        if frame is not None and new_frame is not None:
            try:
                fa = np.asarray(frame, dtype=np.float32)
                fb = np.asarray(new_frame, dtype=np.float32)
                if fa.shape == fb.shape:
                    frame_changed = np.mean(np.abs(fa - fb)) / max(fa.max(), 1.0) > 0.02
                else:
                    frame_changed = True
            except Exception:
                frame_changed = True

        current_levels = getattr(new_obs, "levels_completed", 0) or 0
        level_up = current_levels > prev_levels
        current_score = current_levels / win_levels if win_levels > 0 else 0.0
        score_delta = current_score - prev_score

        try:
            loop.record_result(
                post_frame=new_frame, frame_changed=frame_changed,
                score_delta=score_delta, level_changed=level_up,
                new_level=current_levels + 1 if level_up else 0,
                new_score=current_score,
            )
        except Exception:
            pass

        obs = new_obs
        prev_levels = current_levels
        prev_score = current_score

        state = getattr(obs, "state", None)
        if state in (GameState.WIN, GameState.GAME_OVER):
            break

    final_levels = getattr(obs, "levels_completed", 0) or 0
    final_score = final_levels / win_levels if win_levels > 0 else 0.0
    return final_score, final_levels, actions_taken, loop


print("_cognitive_play() defined")


In [ ]:
# -- Knowledge injection helper ------------------------------------

def _inject_knowledge(causal_map: 'CausalMap', knowledge: dict,
                      game_id: str = None, game_type: str = None):
    """
    Warm-start a CausalMap from bundled JSON knowledge.
    """
    # CausalRule and TileEffect are in the already-loaded causal_map module
    cm_mod = sys.modules['engines.cognition.causal_map']
    CausalRule = getattr(cm_mod, 'CausalRule', None)
    TileEffect = getattr(cm_mod, 'TileEffect', None)

    if not CausalRule or not TileEffect:
        print('WARNING: Could not find CausalRule/TileEffect, skipping injection')
        return

    knowledge_game_id   = knowledge.get('game_id', '')
    knowledge_game_type = knowledge_game_id[:4] if knowledge_game_id else ''
    game_type_match     = bool(game_type and game_type == knowledge_game_type)
    exact_id_match      = bool(game_id and game_id == knowledge_game_id)

    # -- Position-invariant: inject for game_type match ------------
    if game_type_match:
        for rule_data in knowledge.get('rules', []):
            try:
                causal_map._rules.append(CausalRule(
                    rule_type=rule_data['type'],
                    description=rule_data.get('desc', ''),
                    evidence_count=rule_data.get('evidence', 1),
                    confidence=rule_data.get('confidence', 0.5),
                ))
            except Exception:
                pass

        for pos_key, cycle in knowledge.get('color_cycles', {}).items():
            try:
                parts = pos_key.strip('()').split(',')
                pos = (int(parts[0].strip()), int(parts[1].strip()))
                if isinstance(cycle, list):
                    causal_map._color_cycles[pos] = cycle
            except Exception:
                pass

    # -- Tile effects: inject for click-only games (position stable) -
    is_click_only = (
        causal_map.game_id[:4] == knowledge_game_type
        and not any(a in getattr(causal_map, '_known_movement_actions', []) for a in (1, 2, 3, 4))
    )
    inject_effects = exact_id_match or (game_type_match and is_click_only)

    if inject_effects:
        for pos_key, effect_data in knowledge.get('causal_map', {}).items():
            try:
                parts = pos_key.strip('()').split(',')
                pos = (int(parts[0].strip()), int(parts[1].strip()))

                if isinstance(effect_data, dict) and 'transitions' in effect_data:
                    transitions = {}
                    for tpos_key, trans_list in effect_data['transitions'].items():
                        tparts = tpos_key.strip('()').split(',')
                        tpos = (int(tparts[0].strip()), int(tparts[1].strip()))
                        transitions[tpos] = [tuple(t) for t in trans_list]
                    affected = [tuple(a) for a in effect_data.get('affected', [])]
                    te = TileEffect(
                        position=pos,
                        affected=affected,
                        color_transitions=transitions,
                        observation_count=effect_data.get('obs', 1),
                        confidence=effect_data.get('conf', 0.5),
                    )
                    causal_map._effects[pos] = te

                causal_map._explored.add(pos)
                causal_map._all_positions.add(pos)
            except Exception:
                pass

    # -- Position-specific: exact game_id match only ---------------
    if exact_id_match:
        for lvl_str, cells_dict in knowledge.get('win_templates', {}).items():
            try:
                lvl = int(lvl_str)
                cells = {}
                for pos_key, color in cells_dict.items():
                    parts = pos_key.strip('()').split(',')
                    cells[(int(parts[0].strip()), int(parts[1].strip()))] = int(color)
                if cells:
                    causal_map._win_templates[lvl] = cells
            except Exception:
                pass

        goal_cells_data = knowledge.get('goal_cells', {})
        if goal_cells_data and not causal_map._win_templates:
            try:
                for pos_key, color in goal_cells_data.items():
                    parts = pos_key.strip('()').split(',')
                    pos = (int(parts[0].strip()), int(parts[1].strip()))
                    causal_map._goal_cells[pos] = int(color)
                causal_map._goal_source = knowledge.get('goal_source', 'injected')
            except Exception:
                pass

    causal_map.apply_win_template(1)


print('_inject_knowledge() defined')


In [ ]:
# -- Knowledge export helper ---------------------------------------

def _export_knowledge(causal_map: 'CausalMap', game_id: str, output_dir: str):
    """
    Serialize learned CausalMap state to JSON for next-run warm-start.
    Exports: tile effects, rules, color cycles, win templates.
    """
    # Tile effects: {pos -> TileEffect}
    effects = {}
    for pos, te in getattr(causal_map, '_effects', {}).items():
        key = f'({pos[0]},{pos[1]})'
        transitions = {}
        for tpos, trans_list in te.color_transitions.items():
            transitions[f'({tpos[0]},{tpos[1]})'] = list(trans_list)
        effects[key] = {
            'affected': list(te.affected),
            'transitions': transitions,
            'obs': te.observation_count,
            'conf': round(te.confidence, 3),
        }

    # Rules
    rules = []
    for r in getattr(causal_map, '_rules', []):
        rules.append({
            'type': r.rule_type,
            'desc': getattr(r, 'description', ''),
            'evidence': getattr(r, 'evidence_count', 1),
            'confidence': round(r.confidence, 3),
        })

    # Color cycles
    color_cycles = {}
    for pos, cycle in getattr(causal_map, '_color_cycles', {}).items():
        color_cycles[f'({pos[0]},{pos[1]})'] = list(cycle)

    # Win templates (H53)
    win_templates = {}
    for lvl, cells in getattr(causal_map, '_win_templates', {}).items():
        win_templates[str(lvl)] = {
            f'({p[0]},{p[1]})': color for p, color in cells.items()
        }

    # Goal cells (for next session's warm-start)
    goal_cells = {}
    for pos, color in getattr(causal_map, '_goal_cells', {}).items():
        goal_cells[f'({pos[0]},{pos[1]})'] = color

    out = {
        'game_id': game_id,
        'causal_map': effects,
        'rules': rules,
        'color_cycles': color_cycles,
        'win_templates': win_templates,
        'goal_cells': goal_cells,
        'goal_source': getattr(causal_map, '_goal_source', ''),
    }

    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f'{game_id}.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)


print('_export_knowledge() defined')

In [ ]:
# -- Main competition loop -----------------------------------------
# Opens ONE scorecard, plays ALL games, closes it.
# Exports learned knowledge to /kaggle/working/ after each game.

T_START = time.time()
T_LIMIT_HOURS = 5.5  # Leave 30 min buffer for Kaggle overhead
T_LIMIT_SECS = T_LIMIT_HOURS * 3600

KNOWLEDGE_EXPORT_DIR = '/kaggle/working/knowledge_export' if KAGGLE else 'competition_knowledge_export'
os.makedirs(KNOWLEDGE_EXPORT_DIR, exist_ok=True)

scorecard_id = arcade.create_scorecard(
    source_url='https://github.com/BitterTruth-AI',
    tags=['bittertruth', 'cognitive', 'solver-free'],
)
print(f'Scorecard: {scorecard_id}')
print(f'Playing {len(games)} games with {T_LIMIT_HOURS}h budget')
print(f'Time per game budget: {T_LIMIT_SECS / max(len(games),1) / 60:.1f} min')
print(f'Knowledge export: {KNOWLEDGE_EXPORT_DIR}')
print()

all_results = []

for i, game_info in enumerate(games):
    elapsed = time.time() - T_START
    remaining = T_LIMIT_SECS - elapsed
    games_left = len(games) - i
    time_per_game = remaining / max(games_left, 1)

    if remaining < 60:
        print(f'TIME LIMIT approaching -- skipping remaining {games_left} games')
        break

    print(f'[{i+1}/{len(games)}] {game_info.game_id} '
          f'(budget: {time_per_game/60:.1f}min, elapsed: {elapsed/60:.1f}min total)')

    knowledge = BUNDLED_KNOWLEDGE.get(game_info.game_id) or \
                BUNDLED_KNOWLEDGE.get(game_info.game_id[:4], {})

    try:
        result = play_game(
            game_info.game_id,
            scorecard_id=scorecard_id,
            knowledge=knowledge,
            verbose=True,
            knowledge_export_dir=KNOWLEDGE_EXPORT_DIR,
            t_budget=min(time_per_game, 600.0),  # Hard cap: 10 min per game
        )
    except Exception as _game_err:
        print(f'  [{game_info.game_id}] GAME ERROR (skipping): {_game_err}')
        result = {'game_id': game_info.game_id, 'score': 0.0, 'levels': 0, 'elapsed': 0}
    all_results.append(result)
    print(f'  RESULT: score={result["score"]:.3f}, '
          f'levels={result.get("levels", 0)}, '
          f'time={result.get("elapsed", 0):.1f}s')

    # Feed learned knowledge back into BUNDLED_KNOWLEDGE so later games benefit.
    # Same game_id gets direct lookup; same game_type prefix gets type-level fallback.
    exported_path = os.path.join(KNOWLEDGE_EXPORT_DIR, f'{game_info.game_id}.json')
    if os.path.exists(exported_path):
        try:
            with open(exported_path) as _ef:
                _new_k = json.load(_ef)
            BUNDLED_KNOWLEDGE[game_info.game_id] = _new_k
            # Also register under game_type prefix as fallback for unseen variants
            _gtype = game_info.game_id[:4]
            if _gtype not in BUNDLED_KNOWLEDGE:
                BUNDLED_KNOWLEDGE[_gtype] = _new_k
            else:
                # Merge winning_sequences from this game into the type-level entry
                _type_k = BUNDLED_KNOWLEDGE[_gtype]
                for _lvl, _seq in _new_k.get('winning_sequences', {}).items():
                    _type_k.setdefault('winning_sequences', {}).setdefault(_lvl, _seq)
        except Exception as _e:
            print(f'    Knowledge reload failed: {_e}')
    print()

# Always write results + parquet even if loop crashed
print('=' * 60)
print(f'DONE. Total time: {(time.time()-T_START)/60:.1f} min')
print(f'Games played: {len(all_results)}/{len(games)}')
avg_score = sum(r["score"] for r in all_results) / max(len(all_results), 1)
print(f'Average score: {avg_score:.4f}')
print()

# -- Write submission.parquet ----------------------------------
try:
    import pandas as pd
    submission_rows = []
    for r in all_results:
        submission_rows.append({
            'game_id':   r.get('game_id', ''),
            'score':     float(r.get('score', 0.0)),
            'levels':    int(r.get('levels', 0)),
        })
    played_ids = {r.get('game_id') for r in all_results}
    for gi in games:
        if gi.game_id not in played_ids:
            submission_rows.append({'game_id': gi.game_id, 'score': 0.0, 'levels': 0})
    submission_df = pd.DataFrame(submission_rows)
    submission_df.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(f'submission.parquet written: {len(submission_df)} rows')
except Exception as _e:
    print(f'submission.parquet write failed: {_e}')

In [ ]:
# -- Final scorecard + submission ----------------------------------
final_scorecard = arcade.close_scorecard(scorecard_id)

if final_scorecard:
    print(f'Final scorecard score: {final_scorecard.score:.4f}')
    print(f'Environments completed: '
          f'{final_scorecard.total_environments_completed}/{final_scorecard.total_environments}')
    print()
    print('Per-game scores:')
    for env_score in final_scorecard.environments:
        print(f'  {env_score.id}: {env_score.score:.3f}')
else:
    print('Warning: scorecard not available (offline mode -- expected)')
    print('Results summary:')
    for r in sorted(all_results, key=lambda x: x['score'], reverse=True):
        print(f'  {r["game_id"]}: score={r["score"]:.3f}')